In [1]:
import json
import os
import random

import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from itertools import combinations
from openai import OpenAI
from dotenv import load_dotenv

from utils.datetime import date_string_to_quarter
from utils.edinet_api import get_doc_name
from utils.ELO import EloRatingSystem, get_num_games
from utils.datapath import (
    edinet_codes_path,
    nikkei_225_path,
    docs_metadata_path,
    wins_signals_path,
    elo_signals_path,
)

In [2]:
load_dotenv()
XAI_API_KEY = os.getenv("XAI_API_KEY")

client = OpenAI(
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
)

In [3]:
df = pd.read_excel(edinet_codes_path)

with open(nikkei_225_path) as f:
    nikkei_225 = json.load(f)

with open(docs_metadata_path) as f:
    docs_metadata = json.load(f)

In [4]:
# TODO:
# Create a dictionary of quarters and document paths - DONE
# Create combinations of all documents in a quarter or random combinations - DONE
# Create winner function - DONE
# Create output dictionary of document -> number of wins (for all combinations) - DONE
# Create a dictionary of (doc1, doc2) -> winner for ELO score (for all combinations and random combinations) - DONE

# Test with random winner, and then with Grok API

In [5]:
# Create a dictionary of quarters and document paths - DONE

quarterly_docs = defaultdict(list)

for doc in docs_metadata:
    period_end_date = doc["periodEnd"]
    period_end_quater = date_string_to_quarter(period_end_date)
    
    directory_path = os.path.join("..", "documents", period_end_quater)

    save_name = get_doc_name(doc)
    output_path = os.path.join(directory_path, save_name)

    quarterly_docs[period_end_quater].append(output_path)

In [6]:
# Create combinations of all documents in a quarter or random combinations

quarterly_combinations = defaultdict(list)

for quarter, docs in quarterly_docs.items():
    quarterly_combinations[quarter] = list(combinations(docs, 2))
    print(quarter, len(docs), len(list(combinations(docs, 2))))

2022-Q4 193 18528
2023-Q1 35 595
2023-Q2 223 24753
2023-Q3 222 24531
2023-Q4 195 18915


In [7]:
def generate_random_pdf_pair(quarter, iterations):
    docs = quarterly_docs[quarter]

    for i in range(iterations):
        selected_pdfs = random.sample(docs, 2)

        yield (
            selected_pdfs[0],
            selected_pdfs[1],
        )

In [8]:
for quarter in quarterly_docs:
    print(quarter)
    for d1, d2 in generate_random_pdf_pair(quarter, 3):
        print(d1, d2)

2022-Q4
../documents/2022-Q4/E01793_アルプスアルパイン株式会社_140_S100Q3Z6.pdf ../documents/2022-Q4/E02163_マツダ株式会社_140_S100Q5ES.pdf
../documents/2022-Q4/E02166_本田技研工業株式会社_140_S100Q6L1.pdf ../documents/2022-Q4/E03611_三井住友トラスト・ホールディングス株式会社_140_S100Q4GQ.pdf
../documents/2022-Q4/E01532_株式会社小松製作所_140_S100Q6IJ.pdf ../documents/2022-Q4/E03556_株式会社千葉銀行_140_S100Q4EN.pdf
2023-Q1
../documents/2023-Q1/E02103_株式会社ＳＵＭＣＯ_140_S100QQH8.pdf ../documents/2023-Q1/E00492_日本たばこ産業株式会社_140_S100QONM.pdf
../documents/2023-Q1/E00393_サッポロホールディングス株式会社_140_S100QPYA.pdf ../documents/2023-Q1/E00932_中外製薬株式会社_140_S100QNEZ.pdf
../documents/2023-Q1/E00395_キリンホールディングス株式会社_140_S100QPV6.pdf ../documents/2023-Q1/E00816_協和キリン株式会社_140_S100QPDH.pdf
2023-Q2
../documents/2023-Q2/E04499_関西電力株式会社_140_S100RMUB.pdf ../documents/2023-Q2/E27633_東急不動産ホールディングス株式会社_140_S100RL9W.pdf
../documents/2023-Q2/E01602_株式会社ジェイテクト_140_S100RLJU.pdf ../documents/2023-Q2/E04187_ヤマトホールディングス株式会社_140_S100RKBV.pdf
../documents/2023-Q2/E01264_ＪＦＥホールディングス株式会社_140_S100RI

In [ ]:
from utils.signal import get_winner
from utils.datapath import get_stock_code_from_path

# Test get_code_from_path()

for quarter, combinations in quarterly_combinations.items():
    for pdf1_path, pdf2_path in combinations:
        codes = [get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)]
        print(codes)
        break

['6506', '7453']
['7453', '9983']
['1928', '6506']
['1928', '6506']
['1928', '3086']


In [ ]:
# Document winners

quarterly_wins = defaultdict(lambda: defaultdict(int))

for quarter, combinations in quarterly_combinations.items():
    for pdf1_path, pdf2_path in combinations:
        winner = get_winner(None, pdf1_path, pdf2_path)
        codes = [get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)]
        winner_name = codes[winner - 1]
        quarterly_wins[quarter][winner_name] += 1

In [ ]:
# Battle outcomes
quarterly_battle_outcomes = defaultdict(list)

for quarter in quarterly_docs:
    for pdf1_path, pdf2_path in generate_random_pdf_pair(quarter, 2000):
        winner = get_winner(None, pdf1_path, pdf2_path)
        codes = [get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)]
        quarterly_battle_outcomes[quarter].append(tuple(codes + [winner]))

In [13]:
# Create quarterly signals from quarterly wins
quarterly_signals_wins = {}

for quarter, docs in quarterly_docs.items():
    num_docs = len(docs)
    quarterly_signals_wins[quarter] = {
        code: wins / num_docs for code, wins in quarterly_wins[quarter].items()
    }

In [14]:

with open(wins_signals_path, 'w', encoding='utf-8') as f:
    json.dump(quarterly_signals_wins, f)

In [15]:
# ELO carried over quarters!

quarterly_signals_elo = {}
elo_system = EloRatingSystem()

In [ ]:
# Battle outcomes

for quarter, docs in quarterly_docs.items():
    for pdf1_path, pdf2_path in generate_random_pdf_pair(
        quarter, get_num_games(len(docs))
    ):
        winner = get_winner(None, pdf1_path, pdf2_path)
        companyA_code, companyB_code = [
            get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)
        ]
        elo_system.update_ratings(companyA_code, companyB_code, winner)

    quarterly_signals_elo[quarter] = elo_system.get_all_ratings()

In [17]:
with open(elo_signals_path, 'w', encoding='utf-8') as f:
    json.dump(quarterly_signals_elo, f)

In [ ]:
# The signal needs a date!